# EDA 01 — DVF : Transactions Immobilières Parisiennes
**Source** : data.gouv.fr — Demandes de Valeurs Foncières  
**Bronze path** : `data/bronze/dvf_clean/date=YYYY-MM-DD/`  
**Scope** : Paris (dept. 75), Appartements et Maisons

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `id_mutation` | str | Identifiant unique de la transaction |
| `date_mutation` | date | Date de la transaction |
| `valeur_fonciere` | float | Prix de vente (€) |
| `type_local` | str | Appartement / Maison |
| `surface_reelle_bati` | float | Surface bâtie (m²) |
| `nombre_pieces_principales` | int | Nombre de pièces |
| `latitude` / `longitude` | float | Coordonnées GPS |
| `code_postal` | str | Code postal (750xx) |
| `nature_mutation` | str | Type (Vente, etc.) |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
dvf_files = sorted(glob.glob(f"{BRONZE_ROOT}/dvf_clean/**/*.parquet", recursive=True))
print(f"Fichiers DVF trouvés : {len(dvf_files)}")
for f in dvf_files[:5]:
    print(" ", f)


In [ ]:
if dvf_files:
    df = pd.concat([pd.read_parquet(f) for f in dvf_files], ignore_index=True)
else:
    # Données synthétiques pour démonstration si fichiers absents
    rng = np.random.default_rng(42)
    n = 10000
    arrondissements = rng.integers(1, 21, n)
    df = pd.DataFrame({
        "id_mutation":               [f"MUT{i:06d}" for i in range(n)],
        "date_mutation":             pd.date_range("2019-01-01", periods=n, freq="h"),
        "valeur_fonciere":           rng.uniform(100_000, 3_000_000, n),
        "type_local":                rng.choice(["Appartement", "Maison"], n, p=[0.95, 0.05]),
        "surface_reelle_bati":       rng.uniform(20, 200, n),
        "nombre_pieces_principales": rng.integers(1, 7, n),
        "code_postal":               [f"750{arr:02d}" for arr in arrondissements],
        "nature_mutation":           "Vente",
        "latitude":                  48.8566 + rng.uniform(-0.08, 0.08, n),
        "longitude":                 2.3522  + rng.uniform(-0.09, 0.09, n),
        "ingested_at":               pd.Timestamp("now", tz="UTC"),
    })
    df["arrondissement"] = arrondissements
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

df["date_mutation"] = pd.to_datetime(df["date_mutation"])
df["prix_m2"] = df["valeur_fonciere"] / df["surface_reelle_bati"]
print(f"Shape : {df.shape}")
df.head(3)


In [ ]:
# ── Aperçu général ──────────────────────────────────────────────────────────
print("=== Statistiques descriptives ===")
display(df[["valeur_fonciere","surface_reelle_bati","prix_m2","nombre_pieces_principales"]].describe().round(2))
print(f"\nValeurs manquantes :")
display(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nRépartition par type :")
display(df["type_local"].value_counts())


In [ ]:
# ── Distribution des prix au m² ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prices = df.loc[df["prix_m2"].between(1000, 30000), "prix_m2"]
axes[0].hist(prices, bins=60, color="#2196F3", edgecolor="white", linewidth=0.3)
axes[0].set_title("Distribution des prix au m²")
axes[0].set_xlabel("€/m²")
axes[0].set_ylabel("Nombre de transactions")
axes[0].axvline(prices.median(), color="red", linestyle="--", label=f"Médiane : {prices.median():,.0f} €/m²")
axes[0].legend()

axes[1].hist(np.log1p(prices), bins=60, color="#FF9800", edgecolor="white", linewidth=0.3)
axes[1].set_title("Distribution log(prix au m²)")
axes[1].set_xlabel("log(€/m²)")
plt.tight_layout()
plt.show()


In [ ]:
# ── Prix médian par arrondissement ──────────────────────────────────────────
if "arrondissement" not in df.columns:
    df["arrondissement"] = df["code_postal"].str[-2:].astype(int)

arr_stats = (
    df[df["prix_m2"].between(1000, 30000)]
    .groupby("arrondissement")["prix_m2"]
    .agg(["median", "count"])
    .rename(columns={"median": "prix_median_m2", "count": "nb_transactions"})
    .sort_values("prix_median_m2", ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(arr_stats)))
bars = ax.bar(arr_stats.index.astype(str), arr_stats["prix_median_m2"], color=colors)
ax.set_xlabel("Arrondissement")
ax.set_ylabel("Prix médian (€/m²)")
ax.set_title("Prix médian au m² par arrondissement parisien")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
display(arr_stats.round(0))


In [ ]:
# ── Évolution temporelle ────────────────────────────────────────────────────
df_time = (
    df[df["prix_m2"].between(1000, 30000)]
    .set_index("date_mutation")
    .resample("Q")["prix_m2"]
    .median()
    .reset_index()
)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_time["date_mutation"], df_time["prix_m2"], marker="o", color="#2196F3", linewidth=2)
ax.fill_between(df_time["date_mutation"], df_time["prix_m2"], alpha=0.15, color="#2196F3")
ax.set_title("Évolution trimestrielle du prix médian au m² — Paris")
ax.set_ylabel("€/m² (médiane)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
plt.tight_layout()
plt.show()


In [ ]:
# ── Distribution des surfaces ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
surface = df.loc[df["surface_reelle_bati"].between(10, 300), "surface_reelle_bati"]
axes[0].hist(surface, bins=60, color="#9C27B0", edgecolor="white", linewidth=0.3)
axes[0].set_title("Distribution des surfaces bâties")
axes[0].set_xlabel("Surface (m²)")
axes[0].axvline(surface.median(), color="red", linestyle="--", label=f"Médiane : {surface.median():.0f} m²")
axes[0].legend()

pieces_dist = df["nombre_pieces_principales"].value_counts().sort_index()
axes[1].bar(pieces_dist.index.astype(str), pieces_dist.values, color="#4CAF50")
axes[1].set_title("Répartition par nombre de pièces")
axes[1].set_xlabel("Pièces")
axes[1].set_ylabel("Transactions")
plt.tight_layout()
plt.show()
print(f"\nSurface médiane : {surface.median():.1f} m²")
print(f"Corrélation surface ↔ prix_m2 : {df['surface_reelle_bati'].corr(df['prix_m2']):.3f}")


## Conclusions EDA
- Les prix au m² suivent une distribution log-normale avec une forte dispersion inter-arrondissements.
- Les arrondissements centraux (1-8) affichent structurellement les prix les plus élevés.
- Légère corrélation négative surface ↔ prix/m² (les grands appartements sont relativement moins chers au m²).
- Le volume de transactions et la saisonnalité sont des indicateurs importants pour le Silver.
